# ETHOS.RESKit.Wind Workflow

This example demonstrates the ETHOS.RESKit.Wind workflow.

## Preparations
In ETHOS.RESKit.Wind one can simulate each wind turbine individually with its own characteristics.
If you know the turbine model and it is included in RESKit, you can use the actual power curve of the turbines. Otherwise RESKit will create a synthetic power curve for you.
The following specifications need to be given by the user:
- latitude
- longitude
- hub height [m]
- capacity [kW]
- rotor diameter [m] (or power curve)

Workflow:
1. Import required packages
2. Create turbine placements DataFrame with locations and specifications
3. Run the simulation workflow
4. Generate plot


In [ ]:
import reskit as rk
from reskit import data
import pandas as pd
import numpy as np

In [ ]:
# Create Placements DataFrame with turbine locations and specifications

placements = pd.DataFrame(
    {
        "lon": [5.985195, 5.994685, 6.004750],
        "lat": [50.797254, 50.794208, 50.784432],
        "hub_height": [120, 120, 82],
        "capacity": [4000, 4000, 4000],
        "rotor_diam": [130, 150, 136],
    }
)
placements

# Run the simulation workflow
RESKit will run the simulation and create an xarray Dataset with the simulation results for you.
Apart from the capacity_factor, RESKit also includes the turbine specifications and intermediate data used to determine the capacity factor (such as wind speeds, surface pressure etc.)

The workflow's inputs come from the shared ETHOS.Data catalogue. RESKit's maintainers named them once, in the package's collections file (`reskit/data/collections.yaml`): the collection `onshore_wind` maps the handles `era5`, `gwa_100m`, `gwa_50m` and `gwa_200m` to catalogue entries, so this notebook never has to know a file path. `reskit.data.paths()` fetches whatever is missing and returns `{handle: local path}`.

`test=True` selects the small test fixtures, so the example runs in seconds. Drop it to run the identical code on the full data -- the full data is the default, so a forgotten flag never silently simulates on fixtures.

`reskit.data` is a thin layer over one ETHOS.Data handle on `reskit/data/collections.yaml`, and the same file is behind the `reskit-data` command (`reskit-data show`, `reskit-data fetch onshore_wind --test --paths`). Use `reskit-data show` to inspect the selected catalogue. If its pin is unavailable, select a complete catalogue with `RESKIT_DATA_CATALOG` for RESKit or the shared ETHOS configuration. The input-data how-to covers catalogue selection and development staging.

In [ ]:
inputs = data.paths("onshore_wind", test=True)
reskit_xr = rk.wind.wind_era5_PenaSanchezDunkelWinklerEtAl2025(
    placements=placements,
    era5_path=inputs["era5"],
    gwa_100m_path=inputs["gwa_100m"],
    height_scaling_data={50: inputs["gwa_50m"], 200: inputs["gwa_200m"]},
)
reskit_xr

RESKit will output the capacity factor of each location for every hour of the simulated year.

In [ ]:
reskit_xr["capacity_factor"].isel(time=slice(0, 400)).plot.line(x="time")